# 02 Embeddings And Pairs
Embeddings erzeugen/laden, QC, LSPO/ADS Pair-Building.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
USE_STUB_EMBEDDINGS = False
PREFER_PRECOMPUTED_ADS = True
DEVICE = "auto"


ROOT: /home/ubuntu/Author_Name_Disambiguation


In [2]:
import json
import numpy as np
import pandas as pd

from author_name_disambiguation.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from author_name_disambiguation.common.io_schema import save_parquet
from author_name_disambiguation.common.subset_artifacts import load_subset_mentions
from author_name_disambiguation.features.embed_chars2vec import get_or_create_chars2vec_embeddings
from author_name_disambiguation.features.embed_specter import get_or_create_specter_embeddings
from author_name_disambiguation.approaches.nand.build_pairs import assign_lspo_splits, build_pairs_within_blocks, write_pairs

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
MODEL = CTX["MODEL"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "02_run_consistency.json",
    extras={"notebook": "02_embeddings_and_pairs", "latest_context_path": str(latest_context_path)},
)

subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
metrics_dir = RUN_DIRS["metrics"]

lspo_mentions, ads_mentions, subset_meta = load_subset_mentions(
    data_cfg=DATA,
    run_dirs=RUN_DIRS,
    run_cfg=RUN,
    run_stage=RUN_STAGE,
    allow_legacy=True,
    sampler_version="v2",
)
if subset_meta.source == "shared":
    print("Loaded shared subsets:")
else:
    print("Loaded legacy run-local subsets:")
print("  LSPO <-", subset_meta.lspo_path)
print("  ADS  <-", subset_meta.ads_path)
print("Subset tag:", subset_meta.identity.subset_tag)

print("RUN_ID:", RUN_ID)



Loaded shared subsets:
  LSPO <- /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/lspo_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
  ADS  <- /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/ads_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
RUN_ID: smoke_20260213T134837Z_f0fc32b8


In [3]:
rep = MODEL["representation"]

lspo_chars = get_or_create_chars2vec_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
lspo_text = get_or_create_specter_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=16,
    device=DEVICE,
    prefer_precomputed=False,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

ads_chars = get_or_create_chars2vec_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
ads_text = get_or_create_specter_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=32,
    device=DEVICE,
    prefer_precomputed=PREFER_PRECOMPUTED_ADS,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

print("lspo chars/text:", lspo_chars.shape, lspo_text.shape)
print("ads chars/text:", ads_chars.shape, ads_text.shape)


2026-02-13 13:52:07.341358: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-13 13:52:09.080967: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-13 13:52:10.051609: I external/local_xla/xla/stream_executor/gpu/read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-13 13:52:12.011766: I external/local_xla/xla/stream_executor/gpu/read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be a

  9/114 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step   

2026-02-13 13:52:14.333190: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


lspo chars/text: (5000, 50) (5000, 768)
ads chars/text: (5000, 50) (5000, 768)


In [4]:
def emb_qc(name, arr):
    norms = np.linalg.norm(arr, axis=1)
    return {
        "name": name,
        "shape": tuple(arr.shape),
        "nan_count": int(np.isnan(arr).sum()),
        "norm_mean": float(np.mean(norms)),
        "norm_p01": float(np.quantile(norms, 0.01)),
        "norm_p99": float(np.quantile(norms, 0.99)),
    }

pd.DataFrame([
    emb_qc("lspo_chars", lspo_chars),
    emb_qc("lspo_text", lspo_text),
    emb_qc("ads_chars", ads_chars),
    emb_qc("ads_text", ads_text),
])


,name,shape,nan_count,norm_mean,norm_p01,norm_p99
0,lspo_chars,"(5000, 50)",0,3.788135,2.836676,4.592646
1,lspo_text,"(5000, 768)",0,22.516140,21.496834,23.307274
2,ads_chars,"(5000, 50)",0,3.283427,2.402890,4.125783
3,ads_text,"(5000, 768)",0,22.404610,20.955885,23.311216


In [5]:
split_balance_cfg = RUN.get("split_balance", {})
split_assignment_cfg = RUN.get("split_assignment", {})
pair_build_cfg = RUN.get("pair_building", {})

lspo_mentions, split_meta = assign_lspo_splits(
    lspo_mentions,
    seed=int(RUN.get("seed", 11)),
    train_ratio=float(split_assignment_cfg.get("train_ratio", 0.6)),
    val_ratio=float(split_assignment_cfg.get("val_ratio", 0.2)),
    min_neg_val=int(split_balance_cfg.get("min_neg_val", 0)),
    min_neg_test=int(split_balance_cfg.get("min_neg_test", 0)),
    max_attempts=int(split_balance_cfg.get("max_attempts", 1)),
    return_meta=True,
)

lspo_pairs, lspo_pair_meta = build_pairs_within_blocks(
    mentions=lspo_mentions,
    max_pairs_per_block=RUN.get("max_pairs_per_block"),
    seed=int(RUN.get("seed", 11)),
    require_same_split=True,
    labeled_only=False,
    balance_train=True,
    exclude_same_bibcode=bool(pair_build_cfg.get("exclude_same_bibcode", True)),
    return_meta=True,
)

lspo_pairs_path = subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet"
write_pairs(lspo_pairs, lspo_pairs_path)
save_parquet(lspo_mentions, subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet", index=False)

with (metrics_dir / "02_split_balance.json").open("w", encoding="utf-8") as f:
    json.dump(split_meta, f, indent=2)

print("Split balancing:", split_meta)
print("LSPO pair build meta:", lspo_pair_meta)
print("LSPO pairs:", len(lspo_pairs), "->", lspo_pairs_path)


Split balancing: {'status': 'split_balance_degraded', 'attempts': 400, 'min_neg_val': 20, 'min_neg_test': 20, 'split_label_counts': {'train': {'pos': 1701, 'neg': 251, 'labeled_pairs': 1952}, 'val': {'pos': 198, 'neg': 10, 'labeled_pairs': 208}, 'test': {'pos': 193, 'neg': 11, 'labeled_pairs': 204}}}
LSPO pairs: 2364 -> /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/smoke_20260213T134837Z_f0fc32b8/lspo_pairs_smoke.parquet


In [6]:
pair_build_cfg = RUN.get("pair_building", {})

ads_pairs, ads_pair_meta = build_pairs_within_blocks(
    mentions=ads_mentions,
    max_pairs_per_block=RUN.get("max_pairs_per_block"),
    seed=int(RUN.get("seed", 11)),
    require_same_split=False,
    labeled_only=False,
    balance_train=False,
    exclude_same_bibcode=bool(pair_build_cfg.get("exclude_same_bibcode", True)),
    return_meta=True,
)
ads_pairs_path = subset_dir / f"ads_pairs_{RUN_STAGE}.parquet"
write_pairs(ads_pairs, ads_pairs_path)
print("ADS pair build meta:", ads_pair_meta)
print("ADS pairs:", len(ads_pairs), "->", ads_pairs_path)


ADS pairs: 2500 -> /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/smoke_20260213T134837Z_f0fc32b8/ads_pairs_smoke.parquet


In [7]:
def summarize_split_labels(pairs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for split in ["train", "val", "test", "inference", "mixed"]:
        sub = pairs[pairs["split"] == split]
        known = sub[sub["label"].notna()] if "label" in sub.columns else sub.iloc[0:0]
        rows.append({
            "split": split,
            "pairs": int(len(sub)),
            "labeled_pairs": int(len(known)),
            "pos": int((known["label"] == 1).sum()) if len(known) else 0,
            "neg": int((known["label"] == 0).sum()) if len(known) else 0,
        })
    return pd.DataFrame(rows)

split_counts = summarize_split_labels(lspo_pairs)
print("LSPO split label counts")
display(split_counts)

q = lspo_pairs.copy()
if "label" in q.columns:
    known = q[q["label"].notna()]
    pos = int((known["label"] == 1).sum())
    neg = int((known["label"] == 0).sum())
    ratio = pos / max(1, neg)
    print({"labeled_pairs": len(known), "pos": pos, "neg": neg, "pos_neg_ratio": ratio})

print("Pairs per block:")
print(q.groupby("block_key").size().describe())

leak = 0
if "orcid" in lspo_mentions.columns and "split" in lspo_mentions.columns:
    g = lspo_mentions[lspo_mentions["orcid"].notna()].groupby("orcid")["split"].nunique()
    leak = int((g > 1).sum())
print("orcid leakage groups:", leak)

split_label_counts = json.loads(split_counts.to_json(orient="records"))

with (metrics_dir / "02_pairs_qc.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "orcid_leakage_groups": leak,
            "lspo_pairs": int(len(lspo_pairs)),
            "ads_pairs": int(len(ads_pairs)),
            "split_label_counts": split_label_counts,
            "split_balance": split_meta,
            "lspo_pair_build": lspo_pair_meta,
            "ads_pair_build": ads_pair_meta,
        },
        f,
        indent=2,
    )


LSPO split label counts


,split,pairs,labeled_pairs,pos,neg
0,train,1952,1952,1701,251
1,val,208,208,198,10
2,test,204,204,193,11
3,inference,0,0,0,0
4,mixed,0,0,0,0


{'labeled_pairs': 2364, 'pos': 2092, 'neg': 272, 'pos_neg_ratio': 7.6911764705882355}
Pairs per block:
count    2364.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
orcid leakage groups: 0
